In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import importlib
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from plotly.subplots import make_subplots
from plotly import tools
import plotly.offline as pyo
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import glob
from scipy import stats
import uproot
from ROOT import TFile, TEfficiency, TH1D, TGraphAsymmErrors, RDataFrame, TCanvas

c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
ERROR in cling::CIFactory::createCI(): cannot extract standard library include paths!
Invoking:
  LC_ALL=C /Applications/Xcode.app/Contents/Developer/Toolchains/XcodeDefault.xctoolchain/usr/bin/c++ -isysroot;/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX15.1.sdk   -xc++ -E -v /dev/null 2>&1 | sed -n -e '/^.include/,${' -e '/^ \/.*++/p' -e '}'
Results was:
c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
With exit code 0


In [17]:
file = uproot.open("/Users/danielcarber/Documents/ICARUS/purity_mc.root")
print(file.keys())

['events;1', 'events/mc;1', 'events/mc/POT;1', 'events/mc/Livetime;1', 'events/mc/multisigmaTree;2', 'events/mc/multisigmaTree;1', 'events/mc/multisimTree;2', 'events/mc/multisimTree;1', 'events/mc/SelectedNu_Cuts;1', 'events/mc/SelectedCos_PhaseCuts;1', 'events/onbeam;1', 'events/onbeam/POT;1', 'events/onbeam/Livetime;1', 'events/onbeam/SelectedNu_PhaseCuts;1', 'events/offbeam;1', 'events/offbeam/POT;1', 'events/offbeam/Livetime;1', 'events/offbeam/SelectedCos_PhaseCuts;1', 'variations;1', 'variations/reco_electron_energy_GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE_2d;1', 'variations/reco_electron_energy_GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE_2d;1', 'variations/reco_electron_energy_GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE_2d;1', 'variations/reco_electron_energy_GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE_2d;1', 'variations/reco_electron_energy_GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape_2d;1', 'variations/reco_electron_energy_GENIEReWeight_SBN_v1_multisigma_RPA_CCQE_2d;1', 'var

In [18]:
Nu_Purity = file['events/mc/Purity_PhaseCuts;1']
Cosmic_Purity = file['events/mc/SelectedCos_PhaseCuts;1']
Nu_Purity=Nu_Purity.arrays(library='pd')
Cosmic_Purity = Cosmic_Purity.arrays(library='pd')

print(Nu_Purity.keys())
print(sum(Nu_Purity['nu_id']>=0))
mask = (Nu_Purity['nu_id']>=0) & (Nu_Purity['catergory_topology']==0) &


SyntaxError: invalid syntax (767994276.py, line 8)

In [11]:
quality_cuts =  (Nu_Purity['track_containment_cut'] ==1)

signal = Nu_Purity[quality_cuts]
mask  = ((signal['category_topology']==0) | (signal['category_topology']==1)) & (signal['true_proton_energy']>60)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Containement cut: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Containement cut: 0.21%
Number of Signal and Total: 1729, 820268


In [12]:
quality_cuts =  (Nu_Purity['track_containment_cut'] ==1)&\
                (Nu_Purity['fiducial_cut'] ==1)
signal = Nu_Purity[quality_cuts]
mask  = ((signal['category_topology']==0) | (signal['category_topology']==1))& (signal['true_proton_energy']>60)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Fiducial cut: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Fiducial cut: 0.41%
Number of Signal and Total: 1620, 397703


In [13]:
quality_cuts = (Nu_Purity['flash_cut'] ==1)&\
                (Nu_Purity['track_containment_cut'] ==1)&\
                (Nu_Purity['fiducial_cut'] ==1)
signal = Nu_Purity[quality_cuts]
mask  = ((signal['category_topology']==0) | (signal['category_topology']==1))& (signal['true_proton_energy']>60)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Flash cut: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Flash cut: 2.01%
Number of Signal and Total: 1291, 64197


In [14]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)& (Nu_Purity['reco_proton_energy']>60)
signal = Nu_Purity[quality_cuts]
mask  = ((signal['category_topology']==0) | (signal['category_topology']==1))& (signal['true_proton_energy']>60)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of 1eNp: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of 1eNp: 87.28%
Number of Signal and Total: 1002, 1148


In [16]:
quality_cuts = ((Nu_Purity['all_1eNp_cut'] ==1)& (Nu_Purity['reco_electron_axial_spread'] > -0.01)) & (Nu_Purity['reco_proton_energy']>60) 
signal = Nu_Purity[quality_cuts]
mask  = ((signal['category_topology']==0) | (signal['category_topology']==1))& (signal['true_proton_energy']>60)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Axial Spread > 0.02: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Axial Spread > 0.02: 87.28%
Number of Signal and Total: 1002, 1148


In [162]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)&(Nu_Purity['reco_electron_axial_spread'] > -0.01)&(Nu_Purity['reco_electron_dir_spread'] < 0.23)
signal = Nu_Purity[quality_cuts]
mask  =  (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Directional Spread < 0.24: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Directional Spread < 0.24: 83.81%
Number of Signal and Total: 1144, 1365


In [163]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > -0.01)&\
                (Nu_Purity['reco_electron_dir_spread'] < 0.23)&\
                (Nu_Purity['reco_electron_conv_dist'] <7.3)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Conversion Distance > 7.5: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Conversion Distance > 7.5: 85.30%
Number of Signal and Total: 1126, 1320


In [164]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > -0.01) & \
                (Nu_Purity['reco_electron_dir_spread'] < 0.23)&\
                (Nu_Purity['reco_electron_conv_dist'] <7.3)&\
                (Nu_Purity['reco_proton_softmax'] >0.55)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton Softmax > 0.6: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton Softmax > 0.6: 86.32%
Number of Signal and Total: 1117, 1294


In [165]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > -0.01) & \
                (Nu_Purity['reco_electron_dir_spread'] < 0.23)&\
                (Nu_Purity['reco_electron_conv_dist'] <7.3)&\
                (Nu_Purity['reco_proton_softmax'] >0.55)&\
                (Nu_Purity['reco_proton_muon_softmax'] <0.04)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton's Muon Softmax < 0,04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton's Muon Softmax < 0,04: 87.12%
Number of Signal and Total: 1116, 1281


In [156]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > 0.01) & \
                (Nu_Purity['reco_electron_dir_spread'] < 0.15)&\
                (Nu_Purity['reco_electron_conv_dist'] <5.5)&\
                (Nu_Purity['reco_proton_softmax'] >0.65)&\
                (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
                (Nu_Purity['reco_proton_pion_softmax'] <0.35)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton's Pion Softmax < 0.24: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton's Pion Softmax < 0.24: 88.59%
Number of Signal and Total: 1079, 1218


In [157]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > 0.01) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.15)&\
               (Nu_Purity['reco_electron_conv_dist'] <5.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.65)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.35)&\
               (Nu_Purity['reco_electron_softmax'] <0.36)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Softmax < 0.04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Softmax < 0.04: 88.93%
Number of Signal and Total: 1068, 1201


In [159]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > 0.01) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.15)&\
               (Nu_Purity['reco_electron_conv_dist'] <5.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.65)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.35)&\
               (Nu_Purity['reco_electron_softmax'] <0.36)&\
               (Nu_Purity['reco_electron_primary_score'] >0.96)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Primary Score > 0.97: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Primary Score > 0.97: 89.53%
Number of Signal and Total: 1060, 1184
